# S3ANet Automated Experiments
This notebook runs the S3ANet model on all datasets (PaviaU, Salinas, Houston, IndianP), stores the results in proper subfolders, and generates a final Excel report.


## 1. Mount Google Drive and Setup Environment
Assuming your dataset `.mat` files are stored in your Google Drive under a folder named `SACNEet_Data`.
Please modify the `drive_datasets_path` below if they are stored in a different folder.


In [ ]:
from google.colab import drive
import os
import shutil

# Mount Google Drive
drive.mount('/content/drive')

# Ensure the S3ANet project is your current working directory
import os
if not os.path.exists('/content/S3ANET'):
    !git clone https://github.com/SRUJANPATEL3669/S3ANET.git
%cd /content/S3ANET

# Create Data folder
os.makedirs('./Data', exist_ok=True)

# Path to the datasets in your Google Drive (Modify if needed)
drive_datasets_path = '/content/drive/MyDrive/SACNet_Data'

if os.path.exists(drive_datasets_path):
    print("Copying datasets from Google Drive...")
    for file in os.listdir(drive_datasets_path):
        if file.endswith('.mat'):
            shutil.copy(os.path.join(drive_datasets_path, file), './Data/')
    print("Datasets copied successfully!")
else:
    print(f"Warning: The path {drive_datasets_path} does not exist in your Google Drive.")
    print("Please make sure the datasets are present in that folder.")


## 2. Generate Data Samples (`.npy` files)
This step runs `GenSample.py` for datasets 1 to 4 to prepare the training/testing arrays.


In [ ]:
# Run GenSample.py for datasets 1 to 4
for data_id in range(1, 5):
    print(f"\nGenerating samples for Data ID {data_id}...")
    !python GenSample.py --dataID {data_id}
print("\nData generation complete.")


## 3. Run FGSM Attack Experiments and Collect Results
This cell executes `Attack_FGSM_S3ANet.py` for all datasets and captures all metrics:
- **OA, Kappa, AA** — classification accuracy
- **SAM** — mean spectral angle of perturbation (physical-distance metric, degrees)
- **SID** — spectral information divergence (material-identity metric)
- **Physical-consistency rate** — fraction of adversarial pixels passing unmixing round-trip (θ=5°)
- **ASR** — attack success rate (fraction of correctly-classified pixels flipped)

Results are exported to `Attack_Results.xlsx`.


In [ ]:
# Run GenAdvExample.py (Adversarial Example Generation)
# Results will be saved directly to Google Drive
drive_results_path = '/content/drive/MyDrive/S3ANet_Results/'
import os
os.makedirs(drive_results_path, exist_ok=True)

for data_id in range(1, 5):
    print(f'\nGenerating adversarial examples for Data ID {data_id}...')
    !python GenAdvExample.py --dataID {data_id} --save_path_prefix {drive_results_path}
print('\nAdversarial example generation complete.')


In [ ]:
import subprocess
import pandas as pd
import re
import os

datasets = {1: 'PaviaU', 2: 'Salinas', 3: 'Houston', 4: 'IndianP'}
all_results = []

drive_results_path = '/content/drive/MyDrive/S3ANet_Results/'
os.makedirs(drive_results_path, exist_ok=True)

for data_id, dataset_name in datasets.items():
    print(f"\n================ Running Attack Experiment for {dataset_name} ================")
    
    save_prefix = f"/content/drive/MyDrive/S3ANet_Results/Attack_Results/{dataset_name}/"
    os.makedirs(save_prefix, exist_ok=True)
    
    command = f"python Attack_FGSM_S3ANet.py --dataID {data_id} --save_path_prefix {save_prefix}"
    process = subprocess.Popen(command, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    
    output_lines = []
    for line in process.stdout:
        print(line, end='')
        output_lines.append(line.strip())
    process.wait()
    
    oa, kappa, aa, train_time, test_time, runtime = None, None, None, None, None, None
    producer_a = ""
    sam_val, sid_val, phys_rate, asr_val = None, None, None, None
    
    for line in output_lines:
        match = re.search(r'OA=([\d\.]+),Kappa=([\d\.]+)', line)
        if match:
            oa    = float(match.group(1))
            kappa = float(match.group(2))
        
        if "producerA:" in line:
            producer_a = line.split("producerA:")[1].strip()
        
        match = re.search(r'AA=([\d\.]+)', line)
        if match:
            aa = float(match.group(1))
        
        match = re.search(r'Train_time: ([\d\.]+), Test_time: ([\d\.]+), Runtime: ([\d\.]+)', line)
        if match:
            train_time = float(match.group(1))
            test_time  = float(match.group(2))
            runtime    = float(match.group(3))
        
        match = re.search(r'SAM\s*\(mean spectral angle.*?\)\s*:\s*([\d\.]+)', line)
        if match:
            sam_val = float(match.group(1))
        
        match = re.search(r'SID\s*\(spectral info.*?\)\s*:\s*([\d\.]+)', line)
        if match:
            sid_val = float(match.group(1))
        
        match = re.search(r'Physical-consistency rate.*?:\s*([\d\.]+)', line)
        if match:
            phys_rate = float(match.group(1))
        
        match = re.search(r'ASR\s*\(attack success.*?\)\s*:\s*([\d\.]+)', line)
        if match:
            asr_val = float(match.group(1))
    
    all_results.append({
        'Dataset':                    dataset_name,
        'OA (%)':                     oa,
        'Kappa (%)':                  kappa,
        'AA (%)':                     aa,
        'Producer A':                 producer_a,
        'Train Time (s)':             train_time,
        'Test Time (s)':              test_time,
        'Total Runtime (s)':          runtime,
        'SAM (deg)':                  sam_val,
        'SID':                        sid_val,
        'Physical Consistency Rate':  phys_rate,
        'ASR':                        asr_val,
    })

print("\nAll attack experiments finished. Generating Excel report...")

df_attack = pd.DataFrame(all_results)
excel_path = f'{drive_results_path}Attack_Results.xlsx'
df_attack.to_excel(excel_path, index=False)
print(f"Attack results saved to {excel_path}!")
df_attack


## 4. Clean Test — S3ANet on Raw (Unperturbed) Data
This section trains **S3ANet** from scratch on each dataset and evaluates it on the **clean (unperturbed) test set** using `Test_Clean_S3ANet.py`.

Metrics reported:
- **OA, Kappa, AA** — classification accuracy
- **SAM = 0** — reference (no perturbation)
- **SID = 0** — reference (no perturbation)
- **Physical-consistency rate** — baseline unmixing round-trip quality of the clean image (θ=5°)
- **ASR = 0** — reference (no attack applied)

Results are exported to `Clean_Results.xlsx`.


In [ ]:
import subprocess
import pandas as pd
import re
import os

datasets = {1: 'PaviaU', 2: 'Salinas', 3: 'Houston', 4: 'IndianP'}
clean_results = []

drive_results_path = '/content/drive/MyDrive/S3ANet_Results/'
os.makedirs(drive_results_path, exist_ok=True)

for data_id, dataset_name in datasets.items():
    print(f'\n================ Clean Test for {dataset_name} ================')

    save_prefix = f'{drive_results_path}Clean_Results/{dataset_name}/'
    os.makedirs(save_prefix, exist_ok=True)

    command = f'python Test_Clean_S3ANet.py --dataID {data_id} --save_path_prefix {save_prefix}'
    process = subprocess.Popen(
        command, shell=True,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
    )

    output_lines = []
    for line in process.stdout:
        print(line, end='')
        output_lines.append(line.strip())
    process.wait()

    oa, kappa, aa = None, None, None
    producer_a_list = []
    sam_val, sid_val, phys_rate, asr_val = None, None, None, None

    for line in output_lines:
        m = re.search(r'OA\s*:\s*([\d\.]+)', line)
        if m: oa = float(m.group(1))

        m = re.search(r'Kappa\s*:\s*([\d\.]+)', line)
        if m: kappa = float(m.group(1))

        m = re.search(r'AA\s*:\s*([\d\.]+)', line)
        if m: aa = float(m.group(1))

        m = re.search(r'Class\s*(\d+)\s*:\s*([\d\.]+)', line)
        if m: producer_a_list.append(f'C{m.group(1)}={m.group(2)}%')

        m = re.search(r'SAM\s*\(mean spectral angle.*?\)\s*:\s*([\d\.]+)', line)
        if m: sam_val = float(m.group(1))

        m = re.search(r'SID\s*\(spectral info.*?\)\s*:\s*([\d\.]+)', line)
        if m: sid_val = float(m.group(1))

        m = re.search(r'Physical-consistency rate.*?:\s*([\d\.]+)', line)
        if m: phys_rate = float(m.group(1))

        m = re.search(r'ASR\s*\(attack success.*?\)\s*:\s*([\d\.]+)', line)
        if m: asr_val = float(m.group(1))

    clean_results.append({
        'Dataset':                    dataset_name,
        'OA (%)':                     oa,
        'Kappa (%)':                  kappa,
        'AA (%)':                     aa,
        'Producer A':                 ', '.join(producer_a_list),
        'SAM (deg)':                  sam_val,
        'SID':                        sid_val,
        'Physical Consistency Rate':  phys_rate,
        'ASR':                        asr_val,
    })

print('\nAll clean-test experiments finished. Generating Excel report...')

df_clean = pd.DataFrame(clean_results)
clean_excel_path = f'{drive_results_path}Clean_Results.xlsx'
df_clean.to_excel(clean_excel_path, index=False)
print(f'Clean results saved to {clean_excel_path}!')
df_clean


## 5. Clean Test on Perturbed Data
This section runs `Test_Perturbed_S3ANet.py` for all datasets.

The script:
1. Trains S3ANet on clean data
2. Captures clean predictions (for ASR baseline)
3. Generates an FGSM adversarial example
4. Evaluates the **unmodified clean model** on the adversarial image

All 12 metrics are captured and exported to `Perturbed_Results.xlsx`:
- **Clean baseline**: OA, Kappa, AA (model on clean data — reference)
- **Perturbed**: OA, Kappa, AA (same model on adversarial data)
- **SAM** — mean spectral angle of perturbation (degrees)
- **SID** — spectral information divergence
- **Physical-consistency rate** — unmixing round-trip (θ=5°)
- **ASR** — attack success rate
- **Train Time, Attack+Test Time**


In [ ]:
import subprocess
import pandas as pd
import re
import os

datasets = {1: 'PaviaU', 2: 'Salinas', 3: 'Houston', 4: 'IndianP'}
perturbed_results = []

drive_results_path = '/content/drive/MyDrive/S3ANet_Results/'
os.makedirs(drive_results_path, exist_ok=True)

EPSILON = 0.04   # change to match --epsilon used in Section 3

for data_id, dataset_name in datasets.items():
    print(f'\n================ Perturbed Test for {dataset_name} ================')

    save_prefix = f'{drive_results_path}Perturbed_Results/{dataset_name}/'
    os.makedirs(save_prefix, exist_ok=True)

    command = (
        f'python Test_Perturbed_S3ANet.py'
        f' --dataID {data_id}'
        f' --save_path_prefix {save_prefix}'
        f' --epsilon {EPSILON}'
    )
    process = subprocess.Popen(
        command, shell=True,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
    )

    output_lines = []
    for line in process.stdout:
        print(line, end='')
        output_lines.append(line.strip())
    process.wait()

    # ── Parse metrics ─────────────────────────────────────────────────────────
    # Test_Perturbed_S3ANet.py prints (2-space indent in summary block):
    #   [Clean]  OA=92.345%  Kappa=88.456%  AA=85.123%   <- inline progress
    #   [Adv]    OA=60.123%  Kappa=55.678%  AA=50.234%   <- inline progress
    #   (summary block, under section headers)
    #     OA        : 92.345 %   /   OA        : 60.123 %
    #     SAM  (mean spectral angle, deg)  : 2.3147
    #     SID  (spectral info divergence)  : 0.000341
    #     Physical-consistency rate (theta=5.0 deg) : 0.9821  (98.21%)
    #     ASR  (attack success rate)        : 0.6438  (64.38%)
    #     Train time : 12.34s
    #     Attack+test time : 0.56s

    clean_oa, clean_kappa, clean_aa = None, None, None
    adv_oa,   adv_kappa,   adv_aa   = None, None, None
    sam_val, sid_val, phys_rate, asr_val = None, None, None, None
    train_time, test_time = None, None

    in_clean_section     = False
    in_perturbed_section = False

    for line in output_lines:
        # -- Compact inline summary lines from run_one() --
        m = re.search(r'\[Clean\].*OA=([\d\.]+)%.*Kappa=([\d\.]+)%.*AA=([\d\.]+)%', line)
        if m:
            clean_oa, clean_kappa, clean_aa = float(m.group(1)), float(m.group(2)), float(m.group(3))

        m = re.search(r'\[Adv\].*OA=([\d\.]+)%.*Kappa=([\d\.]+)%.*AA=([\d\.]+)%', line)
        if m:
            adv_oa, adv_kappa, adv_aa = float(m.group(1)), float(m.group(2)), float(m.group(3))

        # -- Section-aware OA/Kappa/AA fallback from the printed summary block --
        if 'clean model on CLEAN data' in line:
            in_clean_section, in_perturbed_section = True, False
        if 'clean model on PERTURBED data' in line:
            in_clean_section, in_perturbed_section = False, True
        if 'Spectral' in line and 'Attack' in line:
            in_clean_section, in_perturbed_section = False, False

        m_oa    = re.search(r'OA\s*:\s*([\d\.]+)', line)
        m_kappa = re.search(r'Kappa\s*:\s*([\d\.]+)', line)
        m_aa    = re.search(r'AA\s*:\s*([\d\.]+)', line)
        if in_clean_section:
            if m_oa    and clean_oa    is None: clean_oa    = float(m_oa.group(1))
            if m_kappa and clean_kappa is None: clean_kappa = float(m_kappa.group(1))
            if m_aa    and clean_aa    is None: clean_aa    = float(m_aa.group(1))
        if in_perturbed_section:
            if m_oa    and adv_oa    is None: adv_oa    = float(m_oa.group(1))
            if m_kappa and adv_kappa is None: adv_kappa = float(m_kappa.group(1))
            if m_aa    and adv_aa    is None: adv_aa    = float(m_aa.group(1))

        # -- Spectral / physical metrics (leading 2 spaces) --
        m = re.search(r'SAM\s*\(mean spectral angle.*?\)\s*:\s*([\d\.]+)', line)
        if m: sam_val = float(m.group(1))

        m = re.search(r'SID\s*\(spectral info.*?\)\s*:\s*([\d\.]+)', line)
        if m: sid_val = float(m.group(1))

        m = re.search(r'Physical-consistency rate.*?:\s*([\d\.]+)', line)
        if m: phys_rate = float(m.group(1))

        m = re.search(r'ASR\s*\(attack success.*?\)\s*:\s*([\d\.]+)', line)
        if m: asr_val = float(m.group(1))

        m = re.search(r'Train time\s*:\s*([\d\.]+)', line)
        if m: train_time = float(m.group(1))

        m = re.search(r'Attack\+test time\s*:\s*([\d\.]+)', line)
        if m: test_time = float(m.group(1))

    perturbed_results.append({
        'Dataset':                       dataset_name,
        'Epsilon':                       EPSILON,
        'Clean OA (%)':                  clean_oa,
        'Clean Kappa (%)':               clean_kappa,
        'Clean AA (%)':                  clean_aa,
        'Perturbed OA (%)':              adv_oa,
        'Perturbed Kappa (%)':           adv_kappa,
        'Perturbed AA (%)':              adv_aa,
        'SAM (deg)':                     sam_val,
        'SID':                           sid_val,
        'Physical Consistency Rate':     phys_rate,
        'ASR':                           asr_val,
        'Train Time (s)':                train_time,
        'Attack+Test Time (s)':          test_time,
    })

print('\nAll perturbed-test experiments finished. Generating Excel report...')

df_perturbed = pd.DataFrame(perturbed_results)
perturbed_excel_path = f'{drive_results_path}Perturbed_Results.xlsx'
df_perturbed.to_excel(perturbed_excel_path, index=False)
print(f'Perturbed results saved to {perturbed_excel_path}!')
df_perturbed


## 6. Side-by-Side Comparison: Clean vs Attack vs Perturbed
Merge all three result tables into one unified Excel file with multiple sheets for easy analysis.

| Sheet | Contents |
|---|---|
| `Clean_vs_Attack` | Clean accuracy vs FGSM attack accuracy + spectral metrics |
| `Perturbed_Test` | Clean baseline vs clean-model-on-perturbed + all spectral metrics |
| `Clean_Raw` | Full clean test results |
| `Attack_Raw` | Full attack results |
| `Perturbed_Raw` | Full perturbed test results |


In [ ]:
import pandas as pd

# Load sheets from Drive if DataFrames are not already in memory
try:
    df_attack
    df_clean
    df_perturbed
except NameError:
    drive_results_path = '/content/drive/MyDrive/S3ANet_Results/'
    df_attack    = pd.read_excel(f'{drive_results_path}Attack_Results.xlsx')
    df_clean     = pd.read_excel(f'{drive_results_path}Clean_Results.xlsx')
    df_perturbed = pd.read_excel(f'{drive_results_path}Perturbed_Results.xlsx')

# ── Clean vs Attack comparison ────────────────────────────────────────────────
atk_cols = ['Dataset', 'OA (%)', 'Kappa (%)', 'AA (%)',
            'SAM (deg)', 'SID', 'Physical Consistency Rate', 'ASR']

df_cmp_atk = df_clean[atk_cols].merge(
    df_attack[atk_cols], on='Dataset', suffixes=(' (Clean)', ' (Attack)')
)

# ── Perturbed test: clean baseline vs model-on-perturbed ─────────────────────
pert_cols = ['Dataset', 'Epsilon',
             'Clean OA (%)', 'Clean Kappa (%)', 'Clean AA (%)',
             'Perturbed OA (%)', 'Perturbed Kappa (%)', 'Perturbed AA (%)',
             'SAM (deg)', 'SID', 'Physical Consistency Rate', 'ASR',
             'Train Time (s)', 'Attack+Test Time (s)']
df_cmp_pert = df_perturbed[pert_cols]

# ── Save all sheets into one Excel workbook ───────────────────────────────────
cmp_path = f'{drive_results_path}Comparison_All.xlsx'
with pd.ExcelWriter(cmp_path) as writer:
    df_cmp_atk.to_excel(writer,   sheet_name='Clean_vs_Attack',  index=False)
    df_cmp_pert.to_excel(writer,  sheet_name='Perturbed_Test',    index=False)
    df_clean.to_excel(writer,     sheet_name='Clean_Raw',         index=False)
    df_attack.to_excel(writer,    sheet_name='Attack_Raw',        index=False)
    df_perturbed.to_excel(writer, sheet_name='Perturbed_Raw',     index=False)

print(f'All comparison tables saved to {cmp_path}')
print('\n── Clean vs Attack ──')
display(df_cmp_atk)
print('\n── Clean Model on Perturbed Data ──')
display(df_cmp_pert)
